# 🏐 Process game videos (batch) — v8
**One-time setup:** create Drive folder `balltime` containing `vbpipe.zip`.
Bookmark this notebook (Runtime type: **T4 GPU**).

**Each game night:** copy all trimmed videos (each starts at first serve) into
Drive/balltime, open this notebook, **Runtime → Run all**. It processes every
video that doesn't have a bundle yet and saves bundles to Drive/balltime/bundles/.

⏱ ~30-40 min per game — a 10-game night takes several sessions. If Colab
disconnects, just **Run all** again: it picks up where it left off.
When done, download the bundles from Drive and import them in the app.


In [ ]:
# CONFIG — defaults are right for trimmed videos
GAME_START = "0:00"   # only change if recordings include warmup
LIMIT = 99            # max videos to process this session (lower it to be safe on free Colab)

In [ ]:
# Cell 1 — setup
print("notebook v8 (2026-07-20: bundles ship the FULL video, no per-rally clips; +GPU guard, +lap)")
from google.colab import drive
drive.mount("/content/drive")
import os, zipfile
D = "/content/drive/MyDrive/balltime"
B = f"{D}/bundles"; os.makedirs(B, exist_ok=True)
assert os.path.exists(f"{D}/vbpipe.zip"), f"missing {D}/vbpipe.zip"
zipfile.ZipFile(f"{D}/vbpipe.zip").extractall("pipeline")
%pip -q install ultralytics lap
%pip -q install git+https://github.com/KaiyangZhou/deep-person-reid.git
%pip -q install -e pipeline
import torch
assert torch.cuda.is_available(), (
    "No GPU on this runtime! Runtime -> Change runtime type -> T4 GPU, then Run all again.")
print("GPU:", torch.cuda.get_device_name(0))
gs = GAME_START.split(":"); GS = int(gs[0])*60 + float(gs[1]) if len(gs) > 1 else float(gs[0])
vids = sorted(f for f in os.listdir(D) if f.lower().endswith((".mp4",".mov",".mkv")))
todo = [v for v in vids
        if not os.path.exists(f"{B}/game_bundle_{os.path.splitext(v)[0]}.zip")][:LIMIT]
print(f"{len(vids)} videos in Drive; {len(todo)} still to process: {todo}")
import json as _j
if os.path.exists(f"{D}/courts_config.json"):
    _m = _j.load(open(f"{D}/courts_config.json"))
    print(f"court calibration: {len(_m.get('per_video', {}))} recordings calibrated ✓")
else:
    print("court calibration: none — using defaults (app → Camera setup)")

In [ ]:
# Cell 2 — the per-video pipeline
import subprocess, json, shutil, glob, sys
sys.path.insert(0, "pipeline")

def run(cmd):
    """Stream child output into the cell (Colab hides subprocess stdout/stderr)."""
    p = subprocess.Popen(list(map(str, cmd)), stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout: print(line, end="")
    if p.wait():
        raise RuntimeError(f"command failed (see output above): {' '.join(map(str, cmd))}")

def ensure_ball_model(video, out):
    ball = f"{D}/ball_model.pt"
    if os.path.exists(ball): return ball
    print("  no saved ball model - training once (slow, first video only)...")
    from vbpipe.ballcv import detect_rally
    g = json.load(open(f"{out}/game.json"))
    arcs = {}
    for ri, r in enumerate(g["rallies"]):
        if r.get("phase") == "warmup": continue
        arcs[str(ri)] = [[p[0], p[1], p[2]] for p in detect_rally(video, r, g["tracklets"], ri)]
    json.dump(arcs, open("arcs.json", "w"))
    from vbpipe.balltrain import build_dataset, train
    train(build_dataset(video, "arcs.json"))
    best = sorted(glob.glob("runs/detect/*/weights/best.pt"))[-1]
    shutil.copy(best, ball)
    print("  ball model saved to Drive")
    return ball

def process(vname):
    video = f"{D}/{vname}"
    name = os.path.splitext(vname)[0]
    out = f"out_{name}"
    print(f"=== {vname} ===")
    court = []
    cc = f"{D}/courts_config.json"
    if os.path.exists(cc):
        m = json.load(open(cc))
        key = os.path.splitext(vname)[0]
        geo = (m.get("per_video") or {}).get(key) or m.get("_default")
        if geo:
            json.dump(geo, open(f"court_{key}.json", "w"))
            court = ["--court", f"court_{key}.json"]
            print(f"  court calibration: {'per-video' if key in (m.get('per_video') or {}) else 'default'}")
    elif os.path.exists(f"{D}/court_config.json"):
        court = ["--court", f"{D}/court_config.json"]
    run(["python", "-m", "vbpipe.cli", "full", video, "-o", out,
                    "--device", "cuda"] + court)
    # phase gate (needed before ball training uses game rallies)
    g = json.load(open(f"{out}/game.json"))
    # every detected segment after game start is processed; junk segments get
    # dismissed with one click in the app ("Not a rally")
    for r in g["rallies"]:
        r["phase"] = "game" if r["start"] >= GS else "warmup"
    json.dump(g, open(f"{out}/game.json", "w"))
    ball = ensure_ball_model(video, out)
    run(["python", "-m", "vbpipe.cli", "plays", video, "-o", out,
                    "--game-start", str(GS), "--ball-model", ball,
                    "--eval-clips", "0"] + court)
    # v8: ship the FULL video (app plays rallies via media fragments).
    # faststart moves the moov atom up front so the browser can scrub instantly.
    subprocess.run(["ffmpeg","-v","error","-i",video,"-c","copy",
                    "-movflags","+faststart",f"{out}/game.mp4","-y"], check=True)
    # bundle straight to Drive
    bpath = f"{B}/game_bundle_{name}.zip"
    with zipfile.ZipFile(bpath + ".part", "w") as z:
        z.write(f"{out}/game.json", "game.json")
        z.write(f"{out}/game.mp4", "game.mp4")
        for f in os.listdir(f"{out}/crops"):
            z.write(f"{out}/crops/{f}", f"crops/{f}")
    os.rename(bpath + ".part", bpath)
    shutil.rmtree(out)
    print(f"  ✓ saved {bpath} ({os.path.getsize(bpath)//1_000_000} MB)")

print("ready")

In [ ]:
# Cell 3 — process everything still to do
for v in todo:
    process(v)
print("\nAll done! Download bundles from Drive/balltime/bundles and import them in the app.")